## Custom Lookup Table Demo

This notebook demonstrates **custom lookup tables** in EZKL: using a user-defined piecewise-linear (PWL) table instead of the built-in Sigmoid lookup. This is useful when you need a custom approximation (e.g. different precision or range) for non-linear activations.

### How to produce the lookup table

The lookup table is a JSON file with three arrays:
- **breakpoints** (length n+1): strictly increasing x-values defining segment boundaries. Segments are `[breakpoints[i], breakpoints[i+1])`.
- **slopes** (length n): for segment i, the linear map is `y = slopes[i] * x + intercepts[i]`.
- **intercepts** (length n): intercept for each segment.

You can generate this file by:
1. Choosing breakpoints that cover your model's activation range (e.g. for sigmoid, a symmetric range like [-6, 6]).
2. Computing the target function (e.g. sigmoid) at each breakpoint.
3. For each segment [a, b], set slope = (f(b)-f(a))/(b-a) and intercept = f(a) - slope*a.

**Caveat:** The PWL must approximate the same function your model expects (e.g. sigmoid). Large deviation can cause soundness issues or proof failure; keep approximation error within your tolerance.

In [ ]:
import json
import math
import os

# Produce a PWL approximation of sigmoid(x) = 1/(1+exp(-x))
def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

breakpoints = [-6.0, -3.0, 0.0, 3.0, 6.0]
n = len(breakpoints) - 1
slopes = []
intercepts = []
for i in range(n):
    a, b = breakpoints[i], breakpoints[i + 1]
    fa, fb = sigmoid(a), sigmoid(b)
    slope = (fb - fa) / (b - a)
    intercept = fa - slope * a
    slopes.append(slope)
    intercepts.append(intercept)

pwl = {"breakpoints": breakpoints, "slopes": slopes, "intercepts": intercepts}
pwl_path = 'pwl_sigmoid.json'
with open(pwl_path, 'w') as f:
    json.dump(pwl, f, indent=2)
print('Saved PWL table to pwl_sigmoid.json')

In [ ]:
try:
    import google.colab
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx"])
except Exception:
    pass

from torch import nn
import ezkl
import os
import json
import torch

class SigmoidModel(nn.Module):
    def __init__(self):
        super(SigmoidModel, self).__init__()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(x)

circuit = SigmoidModel()

In [ ]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')
witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')
pwl_path = os.path.abspath('pwl_sigmoid.json')

In [ ]:
shape = [3]
x = 0.5 * (torch.rand(1, *shape) - 0.5)
circuit.eval()
torch.onnx.export(
    circuit, x, model_path,
    export_params=True, opset_version=10, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
data = dict(input_data=[x.detach().numpy().reshape(-1).tolist()])
json.dump(data, open(data_path, 'w'))

In [ ]:
py_run_args = ezkl.PyRunArgs()
py_run_args.input_visibility = "public"
py_run_args.output_visibility = "public"
py_run_args.param_visibility = "fixed"
py_run_args.custom_lookup_path = pwl_path

res = ezkl.gen_settings(model_path, settings_path, py_run_args=py_run_args)
assert res is True

In [ ]:
cal_path = os.path.join('calibration.json')
cal_data = dict(input_data=[torch.rand(20, *shape).detach().numpy().reshape(-1).tolist()])
json.dump(cal_data, open(cal_path, 'w'))
ezkl.calibrate_settings(cal_path, model_path, settings_path, 'resources')

In [ ]:
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res is True

In [ ]:
res = ezkl.get_srs(settings_path)

In [ ]:
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.isfile(witness_path)

In [ ]:
res = ezkl.setup(compiled_model_path, vk_path, pk_path)
assert res is True
assert os.path.isfile(vk_path)
assert os.path.isfile(pk_path)

In [ ]:
proof_path = os.path.join('test.pf')
res = ezkl.prove(witness_path, compiled_model_path, pk_path, proof_path)
assert os.path.isfile(proof_path)

In [ ]:
res = ezkl.verify(proof_path, settings_path, vk_path)
assert res is True
print('Verified.')